# 35 - Whole-Body Control

## What / Why / How

**What we are trying to do:** Understand whole-body control as prioritized objectives under constraints.

**Why this matters:** Humanoids must balance, move hands, avoid joint limits, and respect contacts at the same time.

**How we will do it:** Solve small least-squares control problems with task weights, joint limits, and secondary posture objectives.

## Whole-Body Control

A humanoid needs to satisfy many objectives at once: balance, hand pose, posture, joint limits, and contact constraints.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
# Toy robot with 6 joints and 3 tasks.
rng = np.random.default_rng(35)
J_hand = rng.normal(0, 1, size=(2, 6))
J_com = rng.normal(0, 0.5, size=(2, 6))
J_posture = np.eye(6)

hand_error = np.array([0.12, -0.05])
com_error = np.array([-0.03, 0.02])
posture_error = np.array([0.0, 0.1, -0.1, 0.0, 0.05, -0.05])

A = np.vstack([10 * J_hand, 20 * J_com, 0.5 * J_posture])
b = np.r_[10 * hand_error, 20 * com_error, 0.5 * posture_error]
dq, *_ = np.linalg.lstsq(A, b, rcond=None)
print("joint velocity command:", dq)
print("hand residual:", np.linalg.norm(J_hand @ dq - hand_error))
print("COM residual:", np.linalg.norm(J_com @ dq - com_error))

In [ ]:
joint_limits = np.array([[-1.0, 1.0]] * 6)
q = np.array([0.9, 0.0, -0.95, 0.2, 0.0, 0.0])
q_next = q + dq
q_safe = np.clip(q_next, joint_limits[:, 0], joint_limits[:, 1])
print("raw next q:", q_next)
print("safe next q:", q_safe)
print("clipped joints:", np.where(np.abs(q_next - q_safe) > 1e-9)[0])

In [ ]:
weights = np.linspace(1, 50, 20)
residuals = []
for w in weights:
    A = np.vstack([10 * J_hand, w * J_com, 0.5 * J_posture])
    b = np.r_[10 * hand_error, w * com_error, 0.5 * posture_error]
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    residuals.append((np.linalg.norm(J_hand @ sol - hand_error), np.linalg.norm(J_com @ sol - com_error)))
residuals = np.array(residuals)
print("first/last residuals:", residuals[0], residuals[-1])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(7, 3))
    plt.plot(weights, residuals[:, 0], label="hand residual")
    plt.plot(weights, residuals[:, 1], label="COM residual")
    plt.xlabel("COM task weight")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add a foot contact task.
2. Change task weights and explain tradeoffs.
3. Replace clipping with a penalty before joint limits.